# DAY 4: Domain-Specific Fine-Tuning (Structured JSON Extraction)

## 🎯 Learning Objectives
This is the **core portfolio fine-tuning pipeline**.
By the end of this day, you will be able to execute and explain:
1. **Domain Data Engineering**: Creating a clean dataset mapping messy, unstructured natural language order text into validated JSON schemas.
2. **Strict Split Strategy & Anti-Leakage**: Partitioning data into train (1,200), validation (150), and held-out test (150) splits with programmatic verification of zero overlap.
3. **End-to-End QLoRA + SFT Pipeline**: Combining 4-bit NF4 base quantization with full-linear LoRA adapter training on a Google Colab T4 GPU.
4. **Training Dynamics & Convergence**: Tracking training and validation loss curves across steps.
5. **Artifact Checkpointing**: Saving the fine-tuned adapter weights and training metadata for evaluation and production serving.

### Step 1: Runtime Environment & Hardware Detection

In [1]:
!pip install -q transformers peft trl datasets accelerate bitsandbytes pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 20.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 16.3 MB/s eta 0:00:00:00:0100:01


In [2]:
import os
import sys
import json
import yaml
import torch
import transformers
import peft
import trl

print("=" * 60)
print("🚀 DAY 4: DOMAIN FINE-TUNING PIPELINE")
print("=" * 60)
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name        : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
print(f"Transformers    : {transformers.__version__}")
print(f"PEFT            : {peft.__version__}")
print(f"TRL             : {trl.__version__}")
print("=" * 60)

🚀 DAY 4: DOMAIN FINE-TUNING PIPELINE
PyTorch Version : 2.11.0+cu128
CUDA Available  : True
GPU Name        : Tesla T4
VRAM            : 14.56 GB
Transformers    : 5.15.0
PEFT            : 0.20.0
TRL             : 1.10.0


### Step 2: Generate Domain-Specific Dataset (with Anti-Leakage Guarantee)

In [4]:
import os
import sys

# 1. Clone repository to Colab VM if not present
if not os.path.exists("/content/Fine-Tuning-PoCs"):
    !git clone https://github.com/sagarsambhwani/Fine-Tuning-PoCs.git

# 2. Change directory into the project root
%cd /content/Fine-Tuning-PoCs/llm-finetuning-production

# 3. Add to Python path so 'src' is discoverable
project_root = os.path.abspath("/content/Fine-Tuning-PoCs/llm-finetuning-production")
if project_root not in sys.path:
    sys.path.append(project_root)


Cloning into 'Fine-Tuning-PoCs'...
remote: Enumerating objects: 304, done.
remote: Counting objects: 100% (304/304), done.
remote: Compressing objects: 100% (182/182), done.
remote: Total 304 (delta 122), reused 292 (delta 110), pack-reused 0 (from 0)
Receiving objects: 100% (304/304), 199.30 KiB | 15.33 MiB/s, done.
Resolving deltas: 100% (122/122), done.
/content/Fine-Tuning-PoCs/llm-finetuning-production


In [5]:
from src.data.dataset_generator import generate_order_dataset, save_dataset_splits

# Generate 1,200 Train, 150 Validation, and 150 Test samples deterministically (seed=42)
train_set, val_set, test_set = generate_order_dataset(
    total_train=1200,
    total_val=150,
    total_test=150,
    seed=42
)

save_dataset_splits(train_set, val_set, test_set, output_dir="data/processed")
print(f"Sample Training Record:\n{json.dumps(train_set[0], indent=2)}")

Saved  1200 samples to data/processed/train.jsonl
Saved   150 samples to data/processed/val.jsonl
Saved   150 samples to data/processed/test.jsonl
Sample Training Record:
{
  "id": "order_00001",
  "instruction": "Extract the order details into structured JSON.",
  "input": "Order confirmation: 1 wireless mice bought by Aarav for a total price of $119, shipping out for Friday.",
  "output": {
    "customer": "Aarav",
    "quantity": 1,
    "product": "wireless mice",
    "amount": 119.0,
    "delivery_day": "Friday"
  },
  "messages": [
    {
      "role": "system",
      "content": "You are an expert entity extraction system. Extract structured order information from the user's text into valid JSON according to the schema: {\"customer\": string, \"quantity\": integer, \"product\": string, \"amount\": float, \"delivery_day\": string}."
    },
    {
      "role": "user",
      "content": "Order confirmation: 1 wireless mice bought by Aarav for a total price of $119, shipping out for Fri

### Step 3: Load Base Model in 4-bit NF4 Quantization

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model, TaskType

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=compute_dtype
    )
    base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)
else:
    base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

# Attach LoRA adapter
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


### Step 4: Configure Training Arguments & Execute SFT

In [10]:
import os
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "models/adapters/qwen-1.5b-order-extractor"
os.makedirs(OUTPUT_DIR, exist_ok=True)

dataset = load_dataset("json", data_files={
    "train": "data/processed/train.jsonl",
    "val": "data/processed/val.jsonl"
})

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,      # Effective batch size = 2 * 4 = 8
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    fp16=(compute_dtype == torch.float16),
    bf16=(compute_dtype == torch.bfloat16),
    optim="paged_adamw_8bit" if torch.cuda.is_available() else "adamw_torch",
    report_to="none",
    seed=42,
    max_length=512                      # Renamed to max_length
)

trainer = SFTTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    processing_class=tokenizer,
)

print("🚀 Starting Supervised Fine-Tuning execution...")
train_result = trainer.train()

# Save final adapter
peft_model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Training finished! Adapter saved to {OUTPUT_DIR}")


Tokenizing train dataset:   0%|          | 0/1200 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1200 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1200 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1200 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

...

Dropping fully masked examples from eval dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


🚀 Starting Supervised Fine-Tuning execution...


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,0.221277,0.209245,0.218825,53108.000000,0.936320
100,0.187893,0.188795,0.192108,106025.000000,0.938839
150,0.183324,0.189227,0.188797,159007.000000,0.938849
200,0.182930,0.183771,0.184747,211967.000000,0.939071
250,0.180359,0.181885,0.181847,264988.000000,0.940048
300,0.175832,0.181277,0.179077,318014.000000,0.939235
350,0.169802,0.179257,0.176712,371051.000000,0.939041
400,0.173595,0.178655,0.179041,423958.000000,0.938906
450,0.177998,0.178609,0.178762,477021.000000,0.939201


✅ Training finished! Adapter saved to models/adapters/qwen-1.5b-order-extractor


### Step 5: Save Training Metrics to File

In [11]:
metrics_file = "reports/training_metrics.json"
os.makedirs("reports", exist_ok=True)

metrics_data = {
    "model_id": MODEL_ID,
    "train_loss": train_result.training_loss,
    "train_runtime_seconds": train_result.metrics.get("train_runtime", 0.0),
    "train_samples_per_second": train_result.metrics.get("train_samples_per_second", 0.0),
    "epochs": 3,
    "effective_batch_size": 8,
    "learning_rate": 2e-4
}

with open(metrics_file, "w", encoding="utf-8") as f:
    json.dump(metrics_data, f, indent=2)
print(f"Metrics saved to {metrics_file}:\n{json.dumps(metrics_data, indent=2)}")

Metrics saved to reports/training_metrics.json:
{
  "model_id": "Qwen/Qwen2.5-1.5B-Instruct",
  "train_loss": 0.237460978296068,
  "train_runtime_seconds": 4108.9594,
  "train_samples_per_second": 0.876,
  "epochs": 3,
  "effective_batch_size": 8,
  "learning_rate": 0.0002
}


In [12]:
from google.colab import files
import shutil

# Zip the adapter folder
shutil.make_archive("qwen_1.5b_order_extractor", 'zip', "models/adapters/qwen-1.5b-order-extractor")

# Download to your local PC
files.download("qwen_1.5b_order_extractor.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
!git config user.name "sagarsambhwani"
!git config user.email "sagar.2001.a20@gmail.com"

In [16]:
!git add models/adapters/qwen-1.5b-order-extractor reports/training_metrics.json
!git commit -m "Save Day 4 fine-tuned adapter weights"
!git push origin main

The following paths are ignored by one of your .gitignore files:
llm-finetuning-production/models
hint: Use -f if you really want to add them.
hint: Turn this message off by running
hint: "git config advice.addIgnoredFile false"
[main c01da93] Save Day 4 fine-tuned adapter weights
 1 file changed, 9 insertions(+)
 create mode 100644 llm-finetuning-production/reports/training_metrics.json
fatal: could not read Username for 'https://github.com': No such device or address


In [17]:
import os

print("📁 Current Notebook Directory:", os.getcwd())

adapter_dir = "models/adapters/qwen-1.5b-order-extractor"
if os.path.exists(adapter_dir):
    print("\n✅ Found Adapter Files:")
    for f in os.listdir(adapter_dir):
        print(f"  • {f}")
else:
    print("\nSearching for adapter on this instance...")
    for root, dirs, files in os.walk("/"):
        if "adapter_model.safetensors" in files:
            print(f"  • Found at: {root}")


📁 Current Notebook Directory: /content/Fine-Tuning-PoCs/llm-finetuning-production

✅ Found Adapter Files:
  • tokenizer.json
  • checkpoint-400
  • README.md
  • adapter_config.json
  • checkpoint-450
  • chat_template.jinja
  • tokenizer_config.json
  • adapter_model.safetensors


In [18]:
import shutil

# Zip the adapter folder directly
zip_output = "/content/qwen_1.5b_order_extractor"
shutil.make_archive(zip_output, 'zip', "/content/Fine-Tuning-PoCs/llm-finetuning-production/models/adapters/qwen-1.5b-order-extractor")

print("✅ Zip archive created at: /content/qwen_1.5b_order_extractor.zip")


✅ Zip archive created at: /content/qwen_1.5b_order_extractor.zip


---
## 🎓 Day 4 Interview Preparation & Concept Mastery

### 🧠 Concepts You Must Know
1. **Data Leakage in LLM Fine-Tuning**: Occurs if prompt templates, entities, or paraphrased test samples are present in the training set, giving a false illusion of generalization.
2. **Effective Batch Size**: $\text{Per-device batch size} \times \text{Gradient Accumulation Steps} \times \text{Number of GPUs}$.
3. **Convergence Diagnostics**: Observing steady decrease in both train loss and validation loss. If validation loss begins increasing while train loss continues dropping, overfitting has occurred.
4. **Checkpoint Management**: `save_total_limit = 2` ensures only the top 2 best/most recent checkpoints are kept, preventing disk overflow in Colab sessions.

### 💼 5 Interview Questions
1. **Q1 (Beginner)**: *How did you choose your learning rate ($2 \times 10^{-4}$) for LoRA?*
   - **Answer**: Standard full fine-tuning typically uses small learning rates ($10^{-5}$ to $5 \times 10^{-5}$) because modifying all parameters can destabilize pretrained representations. In LoRA, since base weights are frozen and adapter matrices are initialized from scratch, higher learning rates ($10^{-4}$ to $3 \times 10^{-4}$) are necessary for fast adapter convergence.
2. **Q2 (Intermediate)**: *How did you guarantee that your evaluation results would not suffer from data leakage?*
   - **Answer**: We partitioned our dataset generation using deterministic seeds and executed programmatic set-intersection checks on prompt text across `train`, `val`, and `test` splits, confirming zero shared input strings before training.
3. **Q3 (Intermediate)**: *Why did we set `weight_decay = 0.01`?*
   - **Answer**: Weight decay applies L2 regularization to adapter parameter matrices $A$ and $B$, preventing large numerical weights and reducing the risk of overfitting.
4. **Q4 (Advanced)**: *What would happen if we trained for 10 epochs instead of 3 on 1,200 examples?*
   - **Answer**: On a structured extraction dataset of 1,200 samples, 10 epochs (12,000 example exposures) would likely lead to severe memorization and loss of generalization to unseen sentence structures.
5. **Q5 (Advanced)**: *How do you monitor training stability in real-time?*
   - **Answer**: By tracking step-wise loss, gradient norm ($\|\nabla\| \le 1.0$), and periodic validation loss evaluations every 50 steps.

### 🛠️ 3 Practical Questions on Your Code
1. *In your code, where are the adapter weights saved?*
2. *What is the effective batch size computed in your TrainingArguments?*
3. *How did you pass evaluation datasets into SFTTrainer?*

### 🏆 Day 4 Hands-On Challenge
> **Challenge**: Add an evaluation callback or inspect `trainer.state.log_history` to plot the training vs. validation loss curve across steps using `matplotlib`.